## Project Overview

College Readiness & Completion Analysis | Classes 2016–2025
Institution: Cristo Rey Jesuit High School
Analyst: [Nathan Rayle] | Academic Data Manager & M.S. Data Science Candidate
Data: Full population, 908 students, Classes of 2016–2025
Last Updated: 5/19/2026

This notebook evaluates whether the school's college readiness designation — GPA ≥ 3.0 and at least one ACT benchmark met — predicts college graduation within six years. Classes of 2016–2019 are the primary analysis cohort. Class of 2020 is included with caveats. Class of 2021 is exclud (no testing data). Classes of 2022–2025 receive projected graduation likelihoods based on the model trained on prior cohorts. Methods include permutation testing, bootstrapped confidence intervals, and logistic regression, interpreted under population-level data assumptions.

See README for full study design, variable definitions, and cohort logic.



In [9]:
import pandas as pd
import numpy as np
 
# ── Load ──────────────────────────────────────────────────────────────────────
# The source file is a tab-delimited .txt. We keep only the columns we need
# and rename them to clean working names.
 
df = pd.read_csv(
    "College Readiness Data.txt",          # ← replace with your actual filename
    sep="\t",
    usecols=[2, 3, 5, 6, 7, 9],   # columns C, D, F, G, H, J (0-indexed)
    names=[
        "Student_Number",          # A — skipped
        "Name",                    # B — skipped
        "Class_Of",                # C ✓
        "GPA",                     # D ✓
        "Test",                    # E — skipped
        "Met_GPA",                 # F ✓
        "Met_Benchmark",           # G ✓
        "College_Ready",           # H ✓
        "Match_Field",             # I — skipped
        "College_Status"           # J ✓
    ],
    header=0                       # row 0 is the original header, drop it
)
 
# shape check
print(df.shape)
print(df.dtypes)
print(df.head())

(1703, 6)
Class_Of          float64
GPA               float64
Met_GPA               str
Met_Benchmark         str
College_Ready         str
College_Status        str
dtype: object
   Class_Of   GPA Met_GPA Met_Benchmark College_Ready       College_Status
0    2016.0  3.27     Yes           Yes           Yes   Graduated (2-Year)
1    2016.0  3.32     Yes            No            No   No Longer Enrolled
2    2016.0  3.32     Yes            No            No   No Longer Enrolled
3    2016.0  3.52     Yes           Yes           Yes   No Longer Enrolled
4    2016.0  3.57     Yes           Yes           Yes  Credential Received


In [10]:
# Data Cleaning


df["Class_Of"] = df["Class_Of"].astype(str).str.replace(".0", "", regex=False)
 
# ── Null Check ────────────────────────────────────────────────────────────────
print("Null counts per column:")
print(df.isnull().sum())
 
print("\nData types:")
print(df.dtypes)
 
print("\nClasses present:")
print(df["Class_Of"].unique())
 
print("\nSample:")
print(df.head())
 

Null counts per column:
Class_Of          795
GPA               795
Met_GPA           795
Met_Benchmark     795
College_Ready     793
College_Status    795
dtype: int64

Data types:
Class_Of              str
GPA               float64
Met_GPA               str
Met_Benchmark         str
College_Ready         str
College_Status        str
dtype: object

Classes present:
<StringArray>
['2016', '2017', '2018', '2019', '2020', '2022', '2023', '2024', '2025', nan]
Length: 10, dtype: str

Sample:
  Class_Of   GPA Met_GPA Met_Benchmark College_Ready       College_Status
0     2016  3.27     Yes           Yes           Yes   Graduated (2-Year)
1     2016  3.32     Yes            No            No   No Longer Enrolled
2     2016  3.32     Yes            No            No   No Longer Enrolled
3     2016  3.52     Yes           Yes           Yes   No Longer Enrolled
4     2016  3.57     Yes           Yes           Yes  Credential Received
